# Task 11 Features and forecasting exercises

**Owner: Arman Hajisafi**

This notebook prepares the inputs every model owner can build on. It keeps
historical training data separate from future prediction inputs and documents
what the forecaster is allowed to know at each date.

Our submitted research question is: **Which forecasting models perform best for
medium to long term electricity demand in New South Wales when evaluated across
seasonal variation, temperature changes, and differing demand levels?**

Start here, then read [the handoff guide](task11_handoff.md).
No model fitting is needed to run this notebook. Original data and teammates'
notebooks are preserved. The only optional writes are new local exports.


In [13]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'src' / 'forecast_data.py').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.forecast_data import (
    FeatureConfig, load_prepared_data, load_school_terms, calendar_features,
    default_splits, build_fold, export_fold, public_holiday_calendar,
    evaluation_frame, predictions_frame,
)
CONFIG = FeatureConfig()
print(CONFIG)


FeatureConfig(demand_timezone='Etc/GMT-10', temperature_timezone='Etc/GMT-10', calendar_timezone='Australia/Sydney', temperature_tolerance_minutes=30, climatology_window_days=3, base_temperature_c=18.0, daily_harmonics=3, weekly_harmonics=2, annual_harmonics=3)


In [14]:
data = load_prepared_data(ROOT, CONFIG)
terms = load_school_terms(ROOT)
quality_summary = pd.Series({
    'rows': len(data),
    'first_timestamp': str(data.index.min()),
    'last_timestamp': str(data.index.max()),
    'unreliable_temperature_rows': int(data.temperature_observed_c.isna().sum()),
    'largest_temperature_match_distance_hours': data.temperature_offset_minutes.abs().max() / 60,
})
display(quality_summary.to_frame('value'))
display(data.loc['2016-07-17 12:30':'2016-07-17 14:30'])


,value
rows,196513
first_timestamp,2010-01-01 00:00:00
last_timestamp,2021-03-18 00:00:00
unreliable_temperature_rows,341
largest_temperature_match_distance_hours,45.0


,TOTALDEMAND,temperature_source_time,temperature_offset_minutes,temperature_observed_c
DATETIME,,,,
2016-07-17 12:30:00,7773.99,2016-07-15 16:30:00,-2640.0,NaN
2016-07-17 13:00:00,7651.00,2016-07-15 16:30:00,-2670.0,NaN
2016-07-17 13:30:00,7582.73,2016-07-15 16:30:00,-2700.0,NaN
2016-07-17 14:00:00,7469.72,2016-07-19 11:00:00,2700.0,NaN
2016-07-17 14:30:00,7486.42,2016-07-19 11:00:00,2670.0,NaN


In [15]:
example_times = pd.to_datetime(['2019-12-31 22:30', '2019-12-31 23:00',
                                '2020-01-28 12:00', '2020-02-01 12:00'])
calendar_example = calendar_features(example_times, terms, CONFIG)
display(calendar_example[['half_hour', 'day_of_week', 'is_public_holiday',
                          'is_school_holiday', 'is_weekend', 'trend_years']])


,half_hour,day_of_week,is_public_holiday,is_school_holiday,is_weekend,trend_years
DATETIME,,,,,,
2019-12-31 22:30:00,47.0,1.0,0.0,1.0,0.0,9.998665
2019-12-31 23:00:00,0.0,2.0,1.0,1.0,0.0,9.998722
2020-01-28 12:00:00,26.0,1.0,0.0,0.0,0.0,10.074129
2020-02-01 12:00:00,26.0,5.0,0.0,0.0,1.0,10.085080


In [16]:
splits = default_splits()
display(splits)


,fold_id,role,train_start,origin,forecast_end_exclusive
0,development_20160101,development,2010-01-01,2016-01-01,2017-01-01
1,development_20160401,development,2010-01-01,2016-04-01,2017-04-01
2,development_20160701,development,2010-01-01,2016-07-01,2017-07-01
3,development_20161001,development,2010-01-01,2016-10-01,2017-10-01
4,development_20170101,development,2010-01-01,2017-01-01,2018-01-01
5,development_20170401,development,2010-01-01,2017-04-01,2018-04-01
6,development_20170701,development,2010-01-01,2017-07-01,2018-07-01
7,development_20171001,development,2010-01-01,2017-10-01,2018-10-01
8,final_test_20190101,final_test,2010-01-01,2019-01-01,2020-01-01
9,final_test_20200101,final_test,2010-01-01,2020-01-01,2021-01-01


In [17]:
split = splits.query("role == 'development'").iloc[0]
fold = build_fold(data, split, terms, weather_mode='analogue', config=CONFIG)
print('Training inputs:', fold.X_train.shape)
print('Training demand:', fold.y_train.shape)
print('Future inputs:', fold.X_future.shape)
print('Historical weather donor:', fold.metadata['analogue_year'])
display(fold.X_future.head(3))
display(fold.future_metadata.head(3))
assert fold.X_train.index.max() < fold.X_future.index.min()
assert 'TOTALDEMAND' not in fold.X_future.columns
assert np.isfinite(fold.X_future.to_numpy()).all()


Training inputs: (105168, 28)
Training demand: (105168,)
Future inputs: (17568, 28)
Historical weather donor: 2015


,half_hour,hour,day_of_week,month,day_of_year,is_weekend,utc_offset_hours,is_public_holiday,is_school_holiday,trend_years,...,weekly_sin_2,weekly_cos_2,annual_sin_1,annual_cos_1,annual_sin_2,annual_cos_2,annual_sin_3,annual_cos_3,cooling_degrees_18c,heating_degrees_18c
DATETIME,,,,,,,,,,,,,,,,,,,,,
2016-01-01 00:00:00,2.0,1.0,4.0,1.0,1.0,0.0,11.0,1.0,1.0,5.998754,...,0.826239,0.563320,0.000715,1.000000,0.001431,0.999999,0.002146,0.999998,2.9,0.0
2016-01-01 00:30:00,3.0,1.0,4.0,1.0,1.0,0.0,11.0,1.0,1.0,5.998811,...,0.846724,0.532032,0.001073,0.999999,0.002146,0.999998,0.003219,0.999995,2.0,0.0
2016-01-01 01:00:00,4.0,2.0,4.0,1.0,1.0,0.0,11.0,1.0,1.0,5.998868,...,0.866025,0.500000,0.001431,0.999999,0.002861,0.999996,0.004292,0.999991,1.4,0.0


,lead_month,temperature_input_c,temperature_source_time,temperature_estimated,leap_day_mapped_to_feb28
DATETIME,,,,,
2016-01-01 00:00:00,1,20.9,2015-01-01 00:00:00,False,False
2016-01-01 00:30:00,1,20.0,2015-01-01 00:30:00,False,False
2016-01-01 01:00:00,1,19.4,2015-01-01 01:00:00,False,False


In [ ]:
pd.set_option('display.max_columns', None)
display(fold.X_train.head())

In [11]:
diagnostic = build_fold(data, split, terms, weather_mode='observed_diagnostic', config=CONFIG)
pd.testing.assert_frame_equal(fold.X_train, diagnostic.X_train)
mode = fold.metadata['weather_mode']
weather_comparison = pd.DataFrame({
    f'{mode}_temperature_c': fold.future_metadata.temperature_input_c,
    'observed_diagnostic_temperature_c': diagnostic.future_metadata.temperature_input_c,
})
display(weather_comparison.head(8))


,half_hour,day_of_week,is_public_holiday,is_school_holiday,is_weekend,trend_years
DATETIME,,,,,,
2019-12-31 22:30:00,47.0,1.0,0.0,1.0,0.0,9.998665
2019-12-31 23:00:00,0.0,2.0,1.0,1.0,0.0,9.998722
2020-01-28 12:00:00,26.0,1.0,0.0,0.0,0.0,10.074129
2020-02-01 12:00:00,26.0,5.0,0.0,0.0,1.0,10.085080


## 7 Use the features in your model notebook - example


```python
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'src' / 'forecast_data.py').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.forecast_data import (
    FeatureConfig, load_prepared_data, load_school_terms, default_splits,
    build_fold, evaluation_frame, predictions_frame, export_fold,
)

CONFIG = FeatureConfig(climatology_window_days=3)
data = load_prepared_data(ROOT, CONFIG)
terms = load_school_terms(ROOT)
splits = default_splits()
split = splits.query("role == 'development'").iloc[0]
fold = build_fold(data, split, terms, weather_mode='analogue', config=CONFIG)
```


```python
model.fit(fold.X_train, fold.y_train)
predictions = model.predict(fold.X_future)

truth = evaluation_frame(data, fold)
comparison = truth[['y_true']].copy()
comparison['y_pred'] = predictions
comparison['error'] = comparison['y_pred'] - comparison['y_true']
display(comparison.head())

output = predictions_frame(fold, predictions, model_name='your_model_name')
```

`predictions` contains your model's predictions.
`evaluation_frame` retrieves actual demand for the same dates. The comparison
puts the two together. Predictions must match the future rows in order and
length, with the same index if supplied as a pandas Series.


## 8 Optional CSV handoff

The functions can be imported directly, so exports are optional. Setting the
switch to `True` writes this development exercise under the ignored
`data/NSW/task11_generated` directory. Future demand is not exported. An existing
export is never overwritten; use a new output location for a revised run.


In [ ]:
EXPORT = False  # Change to True to save the selected fold.

if EXPORT:
    folder = export_fold(fold, ROOT / 'data' / 'NSW' / 'task11_generated')
    print(folder)